# Summary

Using LangGraph evaluation tools, evaluate the CCC rag Bbot.

## Import libraries

In [1]:
import os, sys
import json

from langsmith import wrappers, Client
from pydantic import BaseModel, Field
from openai import OpenAI
from langsmith import traceable
from langchain_openai import ChatOpenAI

from typing_extensions import Annotated, TypedDict

########## Change for rag testing
sys.path.insert(0, "utils")
sys.path.insert(0, "../../interface/utils")
from gcp_tools import download_directory_from_gcs
from authentication import ApiAuthentication

sys.path.insert(0, "../../interface/rag")
import rag_bot as rb

sys.path.insert(0, "test_tools/")
import rag_tester as rt


In [2]:
# Test class
rag_tester = rt.ragTester()


Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin
Downloaded chroma.sqlite3 to data/local_chromadb/chroma.sqlite3


In [3]:
rag_tester.run_test()


/opt/anaconda3/envs/ccc-polasst/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'ccc-test1-0fa9d7a6' at:
https://smith.langchain.com/o/522c4165-fa56-4a3e-93ef-53c25a91a9c2/datasets/8e2d65fc-e887-498b-8213-8b38b0e9bb39/compare?selectedSessions=21af55aa-e3c3-4313-b504-e075828ac662




20it [04:32, 13.63s/it]


In [8]:
dir(rag_tester.experiment_results)

rag_tester.experiment_results.experiment_name



'ccc-test1-0fa9d7a6'

### Get API keys

In [2]:
dot_env_path = "../../data/environment"

# Get creds if needed
if len(dot_env_path) > 0:
    creds = ApiAuthentication(dotenv_path=dot_env_path)

    # LangSmith
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_TRACING"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = creds.apis_configs["LANGCHAIN_API_KEY"]
    # Google
    os.environ["GOOGLE_API_KEY"] = creds.apis_configs["GOOGLE_API_KEY"]
    # OpenAI
    os.environ["OPEN_API_KEY"] = creds.apis_configs["OPENAI_KEY"]


## Establish a Langchain client and OpenAI wrapper

In [3]:
client = Client()
openai_client = wrappers.wrap_openai(OpenAI(api_key=os.environ["OPEN_API_KEY"]))


## Create a LangChain example dataset

In [5]:
# Load questions from a JSON File
data_path = "../../../../Numantic/Archive/ccc-tests/data/qa_ipeds_data"
file_name = "ipeds_qa_1.json"
# json_str = json.dumps(qandas)

with open(os.path.join(data_path, file_name), "r") as file:
    example_qas = json.load(file)

type(example_qas)

In [6]:
### These are commented because once created, the dataset is loaded from LangChain

# inputs = [{"question": input_prompt} for input_prompt, _ in example_qas]
# outputs = [{"answer": output_answer} for _, output_answer in example_qas]
#
# examples = [dict(inputs=i, outputs=o) for i, o in zip(inputs, outputs)]

# Delete dataset in LangSmith
# client.delete_dataset(dataset_name = "ccc-testdata-1")


# Create the dataset and examples in LangSmith
# dataset_name = "ccc-testdata-1"
# dataset = client.create_dataset(dataset_name=dataset_name)
# client.create_examples(
#     dataset_id=dataset.id,
#     examples=examples
# )


## Create target testing function using the CCC RAG bot

In [9]:
# Instantiate a CCC bot
bot = rb.CCCPolicyAssistant(chroma_collection_name="crawl_docs-vai-2",
                            chat_bot_verbose=False,
                            dot_env_path=dot_env_path)

# Create a target function
# def target(inputs: dict) -> dict:
# @traceable()

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # Get the bot response
    # bot.show_conversation(input_message=inputs["question"])

    bot.show_conversation(input_message=question)

    # Combine into a single response
    ai_response = "{}".format(bot.ai_response)

    # retrieved_urls = ["- [{}]({})\n".format(up[0], up[1]) for up in bot.retrieved_urls]
    # retrieved_urls = list(set(retrieved_urls))
    #
    # # Create a single string of retrieved URLs
    # res_phrase = ""
    # if len(retrieved_urls) > 0:
    #     res_phrase = "\n\nThese references might be useful: \n{}".format(" ".join(retrieved_urls))
    #
    # # Combine into a single response
    # ai_response = "{} {}".format(bot.ai_response, res_phrase)

    # Add query result response
    if len(bot.query_data_result) > 0:
        ai_response = "{}\n{}".format(ai_response, bot.query_data_result)

    return {"answer": ai_response, "documents": bot.retrieved_docs}


Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin
Downloaded chroma.sqlite3 to data/local_chromadb/chroma.sqlite3


In [10]:
# Test target function
# inputs[0]
# check = rag_bot(inputs[19]["question"])
# check["documents"]


## Define evaluators

In [12]:
# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = ChatOpenAI(model="gpt-4o",
                        temperature=0,
                        api_key=creds.apis_configs["OPENAI_KEY"]).with_structured_output(CorrectnessGrade,
                                                                                         method="json_schema",
                                                                                         strict=True)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
    QUESTION: {inputs['question']}
    GROUND TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

In [13]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o",
                           temperature=0,
                           api_key=creds.apis_configs["OPENAI_KEY"]).with_structured_output(RelevanceGrade,
                                                                                            method="json_schema",
                                                                                            strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]


In [14]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = ChatOpenAI(model="gpt-4o",
                          temperature=0,
                          api_key=creds.apis_configs["OPENAI_KEY"]).with_structured_output(GroundedGrade,
                                                                                           method="json_schema",
                                                                                           strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]


In [15]:
# groundedness(inputs[0]["question"], outputs[0]["answer"])

In [23]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o",
                                     temperature=0,
                                     api_key=creds.apis_configs["OPENAI_KEY"]).with_structured_output(RetrievalRelevanceGrade,
                                                                                                      method="json_schema",
                                                                                                      strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]
    # return {"score": grade["relevant"], "comment": grade["explanation"]}


## Run test

In [24]:
# Function to read rag bot
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data = "ccc-testdata-1",
    # evaluators = [correctness, groundedness, relevance, retrieval_relevance],
    evaluators = [retrieval_relevance],
    experiment_prefix = "ccc-test1",
    max_concurrency = 1,
    metadata={"version": bot.version}
)




View the evaluation results for experiment: 'ccc-test1-096abd78' at:
https://smith.langchain.com/o/522c4165-fa56-4a3e-93ef-53c25a91a9c2/datasets/8e2d65fc-e887-498b-8213-8b38b0e9bb39/compare?selectedSessions=958ca589-0d3c-40a0-9c67-9063352b4bfe




0it [00:00, ?it/s]Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4.0 seconds as it raised ServiceUnavailable: 503 GOAWAY received; Error code: 0; Debug Text: session_timed_out.
2it [00:20,  9.29s/it]This model can reply with multiple function calls in one response. Please don't rely on `additional_kwargs.function_call` as only the last one will be saved.Use `tool_calls` instead.
This model can reply with multiple function calls in one response. Please don't rely on `additional_kwargs.function_call` as only the last one will be saved.Use `tool_calls` instead.
This model can reply with multiple function calls in one response. Please don't rely on `additional_kwargs.function_call` as only the last one will be saved.Use `tool_calls` instead.
20it [01:41,  5.06s/it]


In [25]:
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()



,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.retrieval_relevance,execution_time,example_id,id
0,How many California community college undergra...,"\nI am sorry, I cannot fulfill this request. T...",[page_content='Overview description of file co...,None,"Using the most recent IPEDS data, 1491.0 under...",True,11.318703,2a13ed60-554a-4a44-97e4-0434ad267a53,d4210146-e4ff-4670-b24d-5d595f6fe056
1,Which California community college awarded the...,"I am sorry, but I do not have access to the da...",[page_content='million students among its 115 ...,None,"Using the most recent IPEDS data, WEST LOS ANG...",False,4.113248,3045c0aa-0109-45f0-baae-692ac5f05db3,d5ee11eb-b571-45c1-9525-a1f9d7d85a2e
2,What is Folsom Lake College?,Folsom Lake College (FLC) is a public communit...,[page_content='Folsom Lake College (FLC) is a ...,None,Folsom Lake College (FLC) is a public communit...,True,4.992603,37665091-e550-47b4-a5bb-7e183d30c530,05cab1fe-b100-4548-985b-87c813674cdb
3,Which California community college has the hig...,\nCERRITOS COLLEGE has the most assistant prof...,[page_content='is the main union for classifie...,None,"Using the most recent IPEDS data, EVERGREEN VA...",True,12.149330,44aa6225-f495-4783-9534-99c02083cbce,1c6ee907-f695-4344-a213-f3b3987e89ed
4,How many California community colleges provide...,While a precise count of California Community ...,[page_content='Certain restrictions apply. For...,None,More than 11 California community college camp...,True,3.835207,543aea83-39e2-4353-ac70-ed5c4f8c5aeb,b770a8e4-0842-4605-a75b-58312ae37c52
5,How many California community colleges partner...,"Currently, 22 California community colleges, r...",[page_content='CDCR suspended most of its in‑p...,None,The California Community College system has a ...,True,3.687820,5ce98a0c-4f82-42c3-bb2f-9a26367bc328,760372cb-1d3d-4e2a-8044-9635fcb54110
6,How many districts are there in the California...,The California Community Colleges system is st...,[page_content='Located Throughout the State. T...,None,The California Community Colleges system compr...,True,3.266098,63d04202-3ba1-4f50-97f7-51a857c0c46d,00552109-a58e-4270-87ab-7713656b8329
7,How much did California Community Colleges spe...,"In the fiscal year 2018-19, California Communi...",[page_content='e 1 .2: CALIFORNIA’S COMMUNITY ...,None,"During the analysis year, California’s Communi...",True,3.910418,7381bd28-3db6-4016-9d56-cf77beba3e08,151369dd-ff35-4c7a-b1c2-405ef8f1af72
8,What is the guided pathways program?,Guided Pathways is an equity-focused framework...,[page_content='with funding provided by the le...,None,Every California Community college is implemen...,True,4.580639,7c420153-4bf1-4ad3-9f8e-613a3fa2306c,892e6116-e826-4f90-92c7-3235ce429d0b
9,Which California community college has the low...,"I am sorry, I do not have access to the specif...","[page_content='per unit (or $1,380 for a full‑...",None,"Using the most recent IPEDS data, SHASTA COLLE...",False,4.556963,86becda6-c725-47c9-9fd5-0bad4108b105,af1ccf64-1cae-403f-8269-1bc10c6fd27c
